# 02m — User Items (inventario real)

Procesa `data/data_raw/user_items.csv` (2.08 GB, 17.4M filas) para extraer
features del **inventario real** del jugador: stats de los ítems, mejoras
invertidas, estilo de build (DPS / defensivo / equilibrado).

**No confundir con `user_items_collection`** (procesado en 02l):
- `user_items_collection` (prefijo `coll_`): álbum/códice. 1 fila/jugador/tipo.
- `user_items` (esta tabla, prefijo `items_`): inventario real. **Múltiples
  instancias del mismo tipo** con stats e inversión propia.

**CSV de entrada (11 cols)**: `_id`, `user_id` (float64 por dtype-inference,
hay que forzar str), `item_definition_excel_id`, `c_base_critical_chance`,
`c_base_attack_enhanced`, `c_base_defense_enhanced`, `enhance_level`,
`tempering_level`, `max_enhance_level`, `updated_at`, `created_at`.

**Hallazgo del sondeo**: la tabla tiene una sección inicial larga de
"catálogo" (filas con `user_id` NaN, fecha 2020-04-30) y una sección final
de inventario real (user_ids hex de 24 chars, fechas recientes, todas las
columnas pobladas). El notebook detecta el corte empíricamente.

**Estrategia**: 2 pasadas chunked.
- Pasada 1 (lightweight, solo `user_id` + `created_at`): D1 cobertura global,
  D2 distribución temporal de filas con/sin user_id.
- Pasada 2 (heavy, USECOLS completo, solo si escenario A/B): carga + filtro
  por sample + features.

**Lógica A/B/C** (decidida por código tras pasada 1):
- A (cobertura sample ≥70% AND stats no-100%-nulas): procesamiento completo
- B (30-70% O ventana temporal limitada): procesar con caveat
- C (<30% O todas stats 100% nulas): descarte, sin parquet, append a
  `decision_descarte.md`

**Outputs (escenario A/B)**:
- `data/data_qc/features_user_items.parquet` (126.253 × 14)
- `informes/fase1_cleaning/user_items/execution_report.md`
- HTML completo + sección enriquecida (logging dual)

**Outputs (escenario C)**:
- `informes/fase1_cleaning/user_items/execution_report.md` (versión descarte)
- Append a `informes/decision_descarte.md`


In [1]:
# [SETUP] Imports y dependencias
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import gc
import time
from pathlib import Path
from datetime import datetime, timedelta

plt.style.use('default')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)


In [2]:
# [SETUP] Paths, constantes y helpers
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase1_cleaning' else Path.cwd()
DATA_RAW = PROJECT_ROOT / 'data' / 'data_raw'
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase1_cleaning' / 'user_items'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
DESCARTE_PATH = PROJECT_ROOT / 'informes' / 'decision_descarte.md'

CSV_INPUT = DATA_RAW / 'user_items.csv'
SAMPLE_IDS_PATH = DATA_QC / 'sample_user_ids_cutoff.parquet'
PARQUET_FEATURES = DATA_QC / 'features_user_items_cutoff.parquet'

REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
SENTINEL_DAYS = 9999
CHUNKSIZE = 500_000

# Umbrales de decisión A/B/C
COV_THRESHOLD_A = 0.70  # ≥70% → escenario A
COV_THRESHOLD_B = 0.30  # 30-70% → B; <30% → C

def clean_user_id(uid):
    if pd.isna(uid):
        return None
    uid = str(uid)
    if uid.startswith('ObjectId(') and uid.endswith(')'):
        return uid[9:-1].replace("'", "")
    return uid

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f"[{tag}] {ts} — {message}"
    steps_log.append(entry)
    print(entry)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"CSV_INPUT existe: {CSV_INPUT.exists()} ({CSV_INPUT.stat().st_size/1e9:.2f} GB)" if CSV_INPUT.exists() else "MISSING")
print(f"REFERENCE_DATE: {REFERENCE_DATE}")


PROJECT_ROOT: /Users/jezquerro/Documents/tfg
CSV_INPUT existe: True (2.08 GB)
REFERENCE_DATE: 2026-04-04 00:00:00+00:00


In [3]:
# [SETUP] Carga sample_user_ids
sample_ids_df = pd.read_parquet(SAMPLE_IDS_PATH)
sample_ids_df['user_id'] = sample_ids_df['user_id'].apply(clean_user_id)
sample_user_ids = set(sample_ids_df['user_id'])
N_SAMPLE = len(sample_user_ids)

sample_example = list(sample_user_ids)[0]
assert len(sample_example) == 24 and not sample_example.startswith('ObjectId'), \
    f"ERROR: user_id no es hex limpio: '{sample_example}'"

print(f"Usuarios en sample: {N_SAMPLE:,}")
log_step('SETUP', f'sample_user_ids: {N_SAMPLE:,} usuarios')
log_step('SETUP', f'REFERENCE_DATE: {REFERENCE_DATE}')


Usuarios en sample: 25,200
[SETUP] 13:14:02 — sample_user_ids: 25,200 usuarios
[SETUP] 13:14:02 — REFERENCE_DATE: 2026-04-04 00:00:00+00:00


## 1. Diagnóstico previo — D1, D2, D3, D4

Pasada 1 (lightweight): solo `user_id` + `created_at`. Cuantifica cuánto del
CSV es catálogo (sin user_id) vs inventario real, y dónde está el corte
temporal.


In [4]:
# [EXEC] Pasada 1 — D1 (cobertura user_id) + D2 (distribución temporal)
t0 = time.time()
USECOLS_PASS1 = ['user_id', 'created_at']

n_total = 0
n_null_uid = 0
n_nonnull_uid = 0
month_counts_with_uid = {}
month_counts_no_uid = {}

reader = pd.read_csv(
    CSV_INPUT,
    usecols=USECOLS_PASS1,
    dtype={'user_id': str},
    chunksize=CHUNKSIZE,
)
for chunk in reader:
    n_total += len(chunk)
    mask_null = chunk['user_id'].isna()
    n_null_uid += int(mask_null.sum())
    n_nonnull_uid += int((~mask_null).sum())

    # Distribución temporal por mes (parsing rápido sólo del prefijo YYYY-MM)
    month = chunk['created_at'].str.slice(0, 7)
    for m, cnt in month[mask_null].value_counts().items():
        month_counts_no_uid[m] = month_counts_no_uid.get(m, 0) + int(cnt)
    for m, cnt in month[~mask_null].value_counts().items():
        month_counts_with_uid[m] = month_counts_with_uid.get(m, 0) + int(cnt)

t_pass1 = time.time() - t0

print(f"=== D1 — Cobertura global de user_id ===")
print(f"Filas totales:            {n_total:,}")
print(f"Filas con user_id:        {n_nonnull_uid:,} ({n_nonnull_uid/n_total*100:.2f}%)")
print(f"Filas con user_id nulo:   {n_null_uid:,} ({n_null_uid/n_total*100:.2f}%)")
print(f"Tiempo pasada 1: {t_pass1:.1f}s")

log_step('ANALYSIS', f'D1 cobertura user_id: {n_nonnull_uid:,}/{n_total:,} ({n_nonnull_uid/n_total*100:.2f}%)')

print(f"\n=== D2 — Distribución temporal (top 10 meses) ===")
all_months = sorted(set(month_counts_with_uid) | set(month_counts_no_uid))
print(f"Meses únicos en CSV: {len(all_months)} (rango {all_months[0]} → {all_months[-1]})")
print(f"\n{'Mes':<10} {'con user_id':>15} {'sin user_id':>15} {'% con uid':>10}")
print("-" * 55)
# Mostrar los 10 meses con más filas (ordenados por total desc)
totals = {m: month_counts_with_uid.get(m, 0) + month_counts_no_uid.get(m, 0) for m in all_months}
top_months = sorted(totals.items(), key=lambda x: -x[1])[:10]
for m, total in top_months:
    with_uid = month_counts_with_uid.get(m, 0)
    no_uid = month_counts_no_uid.get(m, 0)
    pct = with_uid / total * 100 if total else 0
    print(f"{m:<10} {with_uid:>15,} {no_uid:>15,} {pct:>9.1f}%")

log_step('ANALYSIS', f'D2 meses únicos: {len(all_months)}, rango {all_months[0]}..{all_months[-1]}')


=== D1 — Cobertura global de user_id ===
Filas totales:            17,450,439
Filas con user_id:        10,636,591 (60.95%)
Filas con user_id nulo:   6,813,848 (39.05%)
Tiempo pasada 1: 9.7s
[ANALYSIS] 13:14:12 — D1 cobertura user_id: 10,636,591/17,450,439 (60.95%)

=== D2 — Distribución temporal (top 10 meses) ===
Meses únicos en CSV: 73 (rango 2020-04 → 2026-04)

Mes            con user_id     sin user_id  % con uid
-------------------------------------------------------
2023-06             20,191         913,490       2.2%
2022-09             12,331         871,576       1.4%
2025-12            803,047               0     100.0%
2026-01            745,227               0     100.0%
2025-11            674,510               0     100.0%
2026-03            663,521               0     100.0%
2026-02            659,219               0     100.0%
2022-10             10,660         557,288       1.9%
2025-01            501,953               0     100.0%
2025-03            438,691          

In [5]:
# [EXEC] Pasada 2 — Carga filtrada (USECOLS completo) + D3 + D4
USECOLS_FULL = [
    'user_id', 'item_definition_excel_id',
    'c_base_critical_chance', 'c_base_attack_enhanced', 'c_base_defense_enhanced',
    'enhance_level', 'tempering_level', 'max_enhance_level',
    'updated_at', 'created_at',
]
N_COLS_RAW = 11  # incluye _id que no leemos

t0 = time.time()
chunks_kept = []
peak_chunk_mem = 0
n_chunks = 0

reader = pd.read_csv(
    CSV_INPUT,
    usecols=USECOLS_FULL,
    dtype={'user_id': str},
    chunksize=CHUNKSIZE,
)
for chunk in reader:
    n_chunks += 1
    chunk_mem = chunk.memory_usage(deep=True).sum() / 1e6
    if chunk_mem > peak_chunk_mem:
        peak_chunk_mem = chunk_mem
    # Filtrar: user_id no nulo Y en sample
    chunk = chunk.dropna(subset=['user_id'])
    if len(chunk) == 0:
        continue
    chunk['user_id'] = chunk['user_id'].apply(clean_user_id)
    kept = chunk[chunk['user_id'].isin(sample_user_ids)].copy()
    if len(kept) > 0:
        chunks_kept.append(kept)

df = pd.concat(chunks_kept, ignore_index=True) if chunks_kept else pd.DataFrame(columns=USECOLS_FULL)
del chunks_kept
gc.collect()

n_in_sample = len(df)
load_seconds = time.time() - t0

# === D3 — Cobertura del sample ===
n_users_with_items = df['user_id'].nunique() if n_in_sample > 0 else 0
coverage_sample = n_users_with_items / N_SAMPLE if N_SAMPLE else 0

print(f"=== Pasada 2 — Carga filtrada ===")
print(f"Chunks procesados: {n_chunks}")
print(f"Memoria pico chunk: {peak_chunk_mem:.1f} MB")
print(f"Filas tras filtro (user_id no null + en sample): {n_in_sample:,}")
print(f"Tiempo pasada 2: {load_seconds:.1f}s")
print(f"Memoria df filtrado: {df.memory_usage(deep=True).sum()/1e6:.1f} MB")

print(f"\n=== D3 — Cobertura del sample ===")
print(f"Usuarios del sample con ≥1 ítem: {n_users_with_items:,}/{N_SAMPLE:,} ({coverage_sample*100:.2f}%)")

log_step('EXEC', f'Pasada 2: {n_in_sample:,} filas en {load_seconds:.1f}s; pico {peak_chunk_mem:.0f} MB')
log_step('ANALYSIS', f'D3 cobertura sample: {n_users_with_items:,}/{N_SAMPLE:,} ({coverage_sample*100:.2f}%)')

# === D4 — Estado de columnas sospechosas (sobre el subset filtrado) ===
print(f"\n=== D4 — % nulos en columnas sobre subset filtrado al sample ===")
if n_in_sample > 0:
    nulls_per_col = df.isnull().sum()
    nulls_pct = (nulls_per_col / n_in_sample * 100).round(4)
    d4_table = pd.DataFrame({'nulls': nulls_per_col, 'pct_nulls': nulls_pct})
    print(d4_table)

    # Stats columns
    stats_cols = ['c_base_critical_chance', 'c_base_attack_enhanced', 'c_base_defense_enhanced']
    stats_all_null = all(nulls_per_col[c] == n_in_sample for c in stats_cols)
    tempering_pct_null = float(nulls_pct['tempering_level'])

    print(f"\nstats_all_null: {stats_all_null}")
    print(f"tempering_level % nulo: {tempering_pct_null:.2f}%")

    # Guardar nulos crudos del subset filtrado
    nulls_per_col.to_csv(REPORT_DIR / 'nulos_por_columna_raw.csv', header=['nulls'])
else:
    print("Sin filas tras filtro — no hay D4 que reportar")
    stats_all_null = True  # treated as 100% nulo si no hay datos
    tempering_pct_null = 100.0
    pd.Series([], name='nulls').to_csv(REPORT_DIR / 'nulos_por_columna_raw.csv', header=True)

log_step('ANALYSIS', f'D4 stats_all_null={stats_all_null}, tempering_pct_null={tempering_pct_null:.2f}%')


=== Pasada 2 — Carga filtrada ===
Chunks procesados: 35
Memoria pico chunk: 76.0 MB
Filas tras filtro (user_id no null + en sample): 720,477
Tiempo pasada 2: 18.7s
Memoria df filtrado: 109.8 MB

=== D3 — Cobertura del sample ===
Usuarios del sample con ≥1 ítem: 25,200/25,200 (100.00%)
[EXEC] 13:14:31 — Pasada 2: 720,477 filas en 18.7s; pico 76 MB
[ANALYSIS] 13:14:31 — D3 cobertura sample: 25,200/25,200 (100.00%)

=== D4 — % nulos en columnas sobre subset filtrado al sample ===
                          nulls  pct_nulls
user_id                       0        0.0
item_definition_excel_id      0        0.0
c_base_critical_chance        0        0.0
c_base_attack_enhanced        0        0.0
c_base_defense_enhanced       0        0.0
enhance_level                 0        0.0
tempering_level               0        0.0
max_enhance_level             0        0.0
updated_at                    0        0.0
created_at                    0        0.0

stats_all_null: False
tempering_level % nulo

In [6]:
# [ANALYSIS] Decisión A/B/C
if coverage_sample >= COV_THRESHOLD_A and not stats_all_null:
    ESCENARIO = 'A'
    desc_escenario = 'Procesable — cobertura ≥70% y stats pobladas'
elif coverage_sample >= COV_THRESHOLD_B and not stats_all_null:
    ESCENARIO = 'B'
    desc_escenario = 'Procesable con caveat — cobertura 30-70%'
else:
    ESCENARIO = 'C'
    desc_escenario = 'Descarte — cobertura <30% o stats todas nulas'

# Decisión sobre tempering: feature descartada en la fuente tras revisión
# (cuasi-constante en valor 1.0; mean=1.00, std=0.05 en el run inicial).
# Se mantiene aquí solo el reporte de nulidad del raw para D4.
desc_tempering = 'tempering_level: feature descartada (cuasi-constante en valor 1.0 post-cómputo)'

print(f"=" * 60)
print(f"DECISIÓN: ESCENARIO = {ESCENARIO}")
print(f"=" * 60)
print(f"  Cobertura sample: {coverage_sample*100:.2f}% (umbrales: A≥{COV_THRESHOLD_A*100:.0f}%, B≥{COV_THRESHOLD_B*100:.0f}%)")
print(f"  Stats todas nulas: {stats_all_null}")
print(f"  Descripción: {desc_escenario}")
print(f"  {desc_tempering}")

log_step('ANALYSIS', f'Escenario={ESCENARIO} ({desc_escenario})')
log_step('ANALYSIS', desc_tempering)


DECISIÓN: ESCENARIO = A
  Cobertura sample: 100.00% (umbrales: A≥70%, B≥30%)
  Stats todas nulas: False
  Descripción: Procesable — cobertura ≥70% y stats pobladas
  tempering_level: feature descartada (cuasi-constante en valor 1.0 post-cómputo)
[ANALYSIS] 13:14:31 — Escenario=A (Procesable — cobertura ≥70% y stats pobladas)
[ANALYSIS] 13:14:31 — tempering_level: feature descartada (cuasi-constante en valor 1.0 post-cómputo)


## 2. Validación de hipótesis (solo escenarios A/B)

In [7]:
# [ANALYSIS] H4 — updated_at ≡ created_at
if ESCENARIO == 'C':
    print(f'[Saltado — escenario {ESCENARIO}]')
    H4_VALUE = float('nan')
    H4_STATE = 'no_aplica'
else:
    # Parsear timestamps
    df['created_at'] = pd.to_datetime(df['created_at'], utc=True, errors='coerce')
    df['updated_at'] = pd.to_datetime(df['updated_at'], utc=True, errors='coerce')

    # Filtro cutoff: descartar instances posteriores al cutoff
    n_pre_cut = len(df)
    df = df[df['created_at'].notna() & (df['created_at'] <= CUTOFF_DATE)].copy()
    log_step('EXEC', f'Filtro cutoff (created_at <= CUTOFF): {n_pre_cut:,} → {len(df):,}')

    mask_eq = (df['updated_at'] == df['created_at'])
    H4_VALUE = float(mask_eq.mean() * 100)
    n_diverge = int((~mask_eq).sum())
    H4_STATE = 'validada' if H4_VALUE >= 99.0 else 'refutada'
    print(f"updated_at ≡ created_at en {H4_VALUE:.4f}% de filas (divergen: {n_diverge:,})")
    print(f"H4 estado: {H4_STATE}")
    log_step('ANALYSIS', f'H4 updated_at==created_at: {H4_VALUE:.4f}% ({H4_STATE})')


[EXEC] 13:14:31 — Filtro cutoff (created_at <= CUTOFF): 720,477 → 557,090
updated_at ≡ created_at en 65.8465% de filas (divergen: 190,266)
H4 estado: refutada
[ANALYSIS] 13:14:31 — H4 updated_at==created_at: 65.8465% (refutada)


## 3. Feature engineering — 13 features con prefijo `items_` (solo A/B)

Decisión cerrada (justificada en el prompt + post-revisión):

**Volumen e inversión (5)**: `items_total_instances`, `items_unique_definitions`,
`items_redundancy_ratio`, `items_mean_enhance_level`, `items_max_enhance_level`.

**Inversión profunda (1)**: `items_total_enhance_invested`.

**Estilo de build (4)**: `items_mean_attack`, `items_mean_defense`,
`items_mean_critical`, `items_attack_defense_ratio` (>1 ofensivo, <1 defensivo).

**Temporales (3)**: `items_first_item_at`, `items_last_item_at`,
`items_days_since_last_item` (sentinel 9999 si sin ítems).

**Eliminadas tras revisión** (cuasi-constancia / saturación):
- `items_mean_tempering` → cuasi-constante en 1.0 (std=0.05).
- `items_pct_max_enhanced` → 97.9% del sample con valor 0.
- `items_n_max_enhanced` → redundante 1:1 con `pct_max_enhanced`.


In [8]:
# [EXEC] Construir las 13 features
if ESCENARIO == 'C':
    print(f'[Saltado — escenario {ESCENARIO}]')
    features = None
else:
    t0 = time.time()

    # GroupBy una sola vez
    agg = df.groupby('user_id').agg(
        items_total_instances=('item_definition_excel_id', 'size'),
        items_unique_definitions=('item_definition_excel_id', 'nunique'),
        items_mean_enhance_level=('enhance_level', 'mean'),
        items_max_enhance_level=('enhance_level', 'max'),
        items_total_enhance_invested=('enhance_level', 'sum'),
        items_mean_attack=('c_base_attack_enhanced', 'mean'),
        items_mean_defense=('c_base_defense_enhanced', 'mean'),
        items_mean_critical=('c_base_critical_chance', 'mean'),
        items_first_item_at=('created_at', 'min'),
        items_last_item_at=('created_at', 'max'),
    )

    # Reindex con sample
    features = agg.reindex(list(sample_user_ids))

    # Counts → 0 para usuarios sin items
    for col in ['items_total_instances', 'items_unique_definitions',
                'items_total_enhance_invested']:
        features[col] = features[col].fillna(0)

    # Ratios derivadas (NaN si denominador 0)
    features['items_redundancy_ratio'] = np.where(
        features['items_unique_definitions'] > 0,
        features['items_total_instances'] / features['items_unique_definitions'],
        np.nan,
    )

    features['items_attack_defense_ratio'] = np.where(
        (features['items_mean_defense'].notna()) & (features['items_mean_defense'] > 0),
        features['items_mean_attack'] / features['items_mean_defense'],
        np.nan,
    )

    # days_since_last_item: sentinel si NaT
    ref_diff = (CUTOFF_DATE - features['items_last_item_at']).dt.total_seconds() / 86400
    features['items_days_since_last_item'] = ref_diff.where(
        features['items_last_item_at'].notna(),
        SENTINEL_DAYS,
    ).astype('int32')

    # Casts finales
    features['items_total_instances'] = features['items_total_instances'].astype('int32')
    features['items_unique_definitions'] = features['items_unique_definitions'].astype('int32')
    features['items_total_enhance_invested'] = features['items_total_enhance_invested'].astype('int32')
    features['items_redundancy_ratio'] = features['items_redundancy_ratio'].astype('float32')
    features['items_attack_defense_ratio'] = features['items_attack_defense_ratio'].astype('float32')
    features['items_mean_enhance_level'] = features['items_mean_enhance_level'].astype('float32')
    features['items_max_enhance_level'] = features['items_max_enhance_level'].astype('float32')
    features['items_mean_attack'] = features['items_mean_attack'].astype('float32')
    features['items_mean_defense'] = features['items_mean_defense'].astype('float32')
    features['items_mean_critical'] = features['items_mean_critical'].astype('float32')

    # Reordenar
    features = features.reset_index().rename(columns={'index': 'user_id'})
    features = features[[
        'user_id',
        # Volumen (5)
        'items_total_instances', 'items_unique_definitions', 'items_redundancy_ratio',
        'items_mean_enhance_level', 'items_max_enhance_level',
        # Inversión profunda (1)
        'items_total_enhance_invested',
        # Estilo (4)
        'items_mean_attack', 'items_mean_defense', 'items_mean_critical', 'items_attack_defense_ratio',
        # Temporal (3)
        'items_first_item_at', 'items_last_item_at', 'items_days_since_last_item',
    ]]

    print(f"Features generadas en {time.time()-t0:.1f}s")
    print(f"Shape: {features.shape}")
    print(f"Dtypes:")
    print(features.dtypes)

    log_step('EXEC', f'Features generadas: {features.shape[1]-1} en {time.time()-t0:.1f}s')


Features generadas en 0.0s
Shape: (25200, 14)
Dtypes:
user_id                                         str
items_total_instances                         int32
items_unique_definitions                      int32
items_redundancy_ratio                      float32
items_mean_enhance_level                    float32
items_max_enhance_level                     float32
items_total_enhance_invested                  int32
items_mean_attack                           float32
items_mean_defense                          float32
items_mean_critical                         float32
items_attack_defense_ratio                  float32
items_first_item_at             datetime64[us, UTC]
items_last_item_at              datetime64[us, UTC]
items_days_since_last_item                    int32
dtype: object
[EXEC] 13:14:31 — Features generadas: 13 en 0.0s


## 4. Sanity checks (solo A/B)

In [9]:
# [ANALYSIS] Sanity checks (asserts bloqueantes)
if ESCENARIO == 'C':
    print(f'[Saltado — escenario {ESCENARIO}]')
    checks_passed = []
    checks_failed = []
else:
    checks_passed = []
    checks_failed = []
    EXPECTED_COLS = 14  # 1 user_id + 13 features

    # 1. Shape
    try:
        assert features.shape == (N_SAMPLE, EXPECTED_COLS), \
            f"shape != ({N_SAMPLE}, {EXPECTED_COLS}): {features.shape}"
        checks_passed.append(f'Shape == ({N_SAMPLE:,}, {EXPECTED_COLS})')
    except AssertionError as e:
        checks_failed.append(str(e))

    # 2. user_id único
    try:
        assert features['user_id'].is_unique, "user_id no es único"
        checks_passed.append('user_id único')
    except AssertionError as e:
        checks_failed.append(str(e))

    mask_with_items = features['items_total_instances'] > 0

    # 3. unique_definitions <= total_instances
    try:
        assert (features['items_unique_definitions'] <= features['items_total_instances']).all(), \
            'unique_definitions > total_instances'
        checks_passed.append('unique_definitions <= total_instances')
    except AssertionError as e:
        checks_failed.append(str(e))

    # 4. redundancy_ratio >= 1 cuando definido
    try:
        rr = features.loc[mask_with_items, 'items_redundancy_ratio']
        assert (rr >= 1.0 - 1e-6).all(), f'redundancy_ratio<1: min={rr.min()}'
        checks_passed.append('redundancy_ratio >= 1 cuando definido')
    except AssertionError as e:
        checks_failed.append(str(e))

    # 5. max_enhance_level >= mean_enhance_level cuando ambos definidos
    try:
        both = features[features['items_mean_enhance_level'].notna() & features['items_max_enhance_level'].notna()]
        assert (both['items_max_enhance_level'] >= both['items_mean_enhance_level'] - 1e-6).all(), \
            'max_enhance_level < mean_enhance_level en algún usuario'
        checks_passed.append('max_enhance_level >= mean_enhance_level')
    except AssertionError as e:
        checks_failed.append(str(e))

    # 6. days_since_last_item >= 0 cuando no es sentinel
    try:
        ds_active = features.loc[features['items_days_since_last_item'] != SENTINEL_DAYS, 'items_days_since_last_item']
        assert (ds_active >= 0).all(), f'days_since negativo: min={ds_active.min()}'
        checks_passed.append('days_since_last_item >= 0 (excluye SENTINEL)')
    except AssertionError as e:
        checks_failed.append(str(e))

    # 7. days_since == SENTINEL ↔ last_item_at NaT
    try:
        sent_mask = features['items_days_since_last_item'] == SENTINEL_DAYS
        nat_mask = features['items_last_item_at'].isna()
        assert (sent_mask == nat_mask).all(), 'incoherencia SENTINEL ↔ NaT'
        checks_passed.append('days_since==SENTINEL ↔ last_item_at NaT')
    except AssertionError as e:
        checks_failed.append(str(e))

    # 8. Tipos
    try:
        for c in ['items_total_instances', 'items_unique_definitions',
                  'items_total_enhance_invested', 'items_days_since_last_item']:
            assert str(features[c].dtype) == 'int32', f'{c}: {features[c].dtype}'
        for c in ['items_redundancy_ratio', 'items_attack_defense_ratio',
                  'items_mean_enhance_level', 'items_max_enhance_level',
                  'items_mean_attack', 'items_mean_defense', 'items_mean_critical']:
            assert str(features[c].dtype) == 'float32', f'{c}: {features[c].dtype}'
        checks_passed.append('Tipos correctos (int32×4, float32×7, datetime×2)')
    except AssertionError as e:
        checks_failed.append(f'Tipos: {e}')

    print("=== SANITY CHECKS ===")
    for c in checks_passed:
        print(f"  {c}")
    for c in checks_failed:
        print(f"  {c}")

    assert len(checks_failed) == 0, f"BLOQUEO: {len(checks_failed)} checks fallidos"
    print(f"\n{len(checks_passed)}/{len(checks_passed)} checks pasaron")
    log_step('ANALYSIS', f'Sanity checks: {len(checks_passed)} OK')


=== SANITY CHECKS ===
  Shape == (25,200, 14)
  user_id único
  unique_definitions <= total_instances
  redundancy_ratio >= 1 cuando definido
  max_enhance_level >= mean_enhance_level
  days_since_last_item >= 0 (excluye SENTINEL)
  days_since==SENTINEL ↔ last_item_at NaT
  Tipos correctos (int32×4, float32×7, datetime×2)

8/8 checks pasaron
[ANALYSIS] 13:14:31 — Sanity checks: 8 OK


## 5. Validación de features (solo A/B)

In [10]:
# [ANALYSIS] Tipos de datos finales + zeros vs nonzeros
if ESCENARIO == 'C':
    print(f'[Saltado — escenario {ESCENARIO}]')
else:
    print("=== Dtypes finales + distribución de ceros ===")
    for col in features.columns:
        dt = features[col].dtype
        nulls = int(features[col].isnull().sum())
        if pd.api.types.is_numeric_dtype(features[col]):
            zeros = int((features[col] == 0).sum())
            nonzero = len(features) - zeros - nulls
            print(f"  {col:35s} dtype={str(dt):10s} nulls={nulls:>7,} zeros={zeros:>7,} nonzero={nonzero:>7,}")
        else:
            print(f"  {col:35s} dtype={str(dt):10s} nulls={nulls:>7,}")


=== Dtypes finales + distribución de ceros ===
  user_id                             dtype=str        nulls=      0
  items_total_instances               dtype=int32      nulls=      0 zeros=    834 nonzero= 24,366
  items_unique_definitions            dtype=int32      nulls=      0 zeros=    834 nonzero= 24,366
  items_redundancy_ratio              dtype=float32    nulls=    834 zeros=      0 nonzero= 24,366
  items_mean_enhance_level            dtype=float32    nulls=    834 zeros=      0 nonzero= 24,366
  items_max_enhance_level             dtype=float32    nulls=    834 zeros=      0 nonzero= 24,366
  items_total_enhance_invested        dtype=int32      nulls=      0 zeros=    834 nonzero= 24,366
  items_mean_attack                   dtype=float32    nulls=    834 zeros=      0 nonzero= 24,366
  items_mean_defense                  dtype=float32    nulls=    834 zeros=      0 nonzero= 24,366
  items_mean_critical                 dtype=float32    nulls=    834 zeros= 10,952 nonzero= 

In [11]:
# [ANALYSIS] Nulos en features finales
if ESCENARIO == 'C':
    print(f'[Saltado — escenario {ESCENARIO}]')
else:
    nulos_final = features.isnull().sum()
    nulos_final.to_csv(REPORT_DIR / 'nulos_features_final.csv', header=['nulls'])
    print("Nulos guardados en nulos_features_final.csv")
    print(nulos_final)


Nulos guardados en nulos_features_final.csv
user_id                           0
items_total_instances             0
items_unique_definitions          0
items_redundancy_ratio          834
items_mean_enhance_level        834
items_max_enhance_level         834
items_total_enhance_invested      0
items_mean_attack               834
items_mean_defense              834
items_mean_critical             834
items_attack_defense_ratio      834
items_first_item_at             834
items_last_item_at              834
items_days_since_last_item        0
dtype: int64


In [12]:
# [ANALYSIS] Estadísticas descriptivas
if ESCENARIO == 'C':
    print(f'[Saltado — escenario {ESCENARIO}]')
else:
    numeric_cols = features.select_dtypes(include=['number']).columns
    desc = features[numeric_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]).T
    desc.to_csv(REPORT_DIR / 'features_describe.csv')
    print(desc.round(4).to_string())
    log_step('ANALYSIS', 'features_describe.csv guardado')


                                count      mean        std      min       25%       50%       75%       90%        99%         max
items_total_instances         25200.0   22.1067    26.5141   0.0000    6.0000   13.0000   28.0000   52.0000   133.0000    471.0000
items_unique_definitions      25200.0   17.0459    15.4810   0.0000    6.0000   12.0000   24.0000   38.0000    70.0000    144.0000
items_redundancy_ratio        24366.0    1.1499     0.2558   1.0000    1.0000    1.0625    1.2069    1.4000     2.1415      8.1667
items_mean_enhance_level      24366.0    2.5020     2.7758   1.0000    1.1000    1.5000    2.5833    5.2899    13.2792     60.0000
items_max_enhance_level       24366.0    6.6638     7.3699   1.0000    2.0000    4.0000    9.0000   16.0000    30.0000     60.0000
items_total_enhance_invested  25200.0   41.4809    67.4407   0.0000   12.0000   27.0000   50.2500   88.0000   244.0000   3197.0000
items_mean_attack             24366.0  299.2393   789.6566  45.0000  102.3750  131.

In [13]:
# [ANALYSIS] Histogramas de features clave
if ESCENARIO == 'C':
    print(f'[Saltado — escenario {ESCENARIO}]')
else:
    fig, axes = plt.subplots(3, 3, figsize=(15, 10))
    axes = axes.flatten()

    counts_log = [
        'items_total_instances', 'items_unique_definitions', 'items_redundancy_ratio',
        'items_mean_enhance_level', 'items_total_enhance_invested',
    ]
    style = [
        'items_mean_attack', 'items_mean_defense', 'items_attack_defense_ratio',
    ]
    for ax, col in zip(axes[:5], counts_log):
        vals = features[col].dropna()
        vals_pos = vals[vals > 0]
        if len(vals_pos) > 0:
            ax.hist(vals_pos, bins=50, color='#3498db', edgecolor='none')
            ax.set_yscale('log')
        ax.set_title(f"{col}\n(n>0={len(vals_pos):,}, NaN={int(features[col].isna().sum()):,})", fontsize=9)
        ax.set_xlabel(col)

    for ax, col in zip(axes[5:8], style):
        vals = features[col].dropna()
        if len(vals) > 0:
            ax.hist(vals, bins=50, color='#9b59b6', edgecolor='none')
        ax.set_title(f"{col}\n(NaN={int(features[col].isna().sum()):,})", fontsize=9)
        ax.set_xlabel(col)

    # days_since (excluye SENTINEL)
    ax = axes[8]
    vals = features.loc[features['items_days_since_last_item'] != SENTINEL_DAYS, 'items_days_since_last_item']
    ax.hist(vals, bins=50, color='#e67e22', edgecolor='none')
    ax.set_yscale('log')
    ax.set_title(f"items_days_since_last_item\n(excluye SENTINEL: {int((features['items_days_since_last_item']==SENTINEL_DAYS).sum()):,})", fontsize=9)
    ax.set_xlabel('días')

    plt.suptitle('Distribución features items_ (escala log donde aplica)', fontsize=12)
    plt.tight_layout()
    out_png = REPORT_DIR / 'features_distributions.png'
    plt.savefig(out_png, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"Histogramas guardados: {out_png}")
    log_step('ANALYSIS', 'features_distributions.png guardado')


Histogramas guardados: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_items/features_distributions.png
[ANALYSIS] 13:14:32 — features_distributions.png guardado


## 6. Guardado del parquet (solo A/B)

In [14]:
# [EXEC] Guardar parquet + sample_head
if ESCENARIO == 'C':
    print(f'[Saltado — escenario {ESCENARIO}: NO se genera parquet]')
else:
    t0 = time.time()
    features.to_parquet(PARQUET_FEATURES, engine='pyarrow', compression='snappy', index=False)
    size_mb = PARQUET_FEATURES.stat().st_size / 1e6
    print(f"Parquet guardado en {time.time()-t0:.1f}s: {PARQUET_FEATURES} ({size_mb:.2f} MB)")

    features_read = pd.read_parquet(PARQUET_FEATURES)
    assert features_read.shape == features.shape
    print(f"Parquet legible, shape={features_read.shape}")

    features.head(20).to_csv(REPORT_DIR / 'sample_head.csv', index=False)
    print(f"sample_head.csv guardado")

    del df, features_read
    gc.collect()
    log_step('EXEC', f'Parquet guardado: ({features.shape[0]:,}, {features.shape[1]}), {size_mb:.2f} MB')


Parquet guardado en 0.0s: /Users/jezquerro/Documents/tfg/data/data_qc/features_user_items_cutoff.parquet (1.73 MB)
Parquet legible, shape=(25200, 14)
sample_head.csv guardado
[EXEC] 13:14:32 — Parquet guardado: (25,200, 14), 1.73 MB


## 7. Informe de ejecución

In [15]:
# [REPORT] Generar execution_report.md (variantes A/B vs C)
t_total = time.time() - NOTEBOOK_START
t_min = int(t_total // 60)
t_sec = int(t_total % 60)
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

lines = []

if ESCENARIO == 'C':
    # ─── Plantilla de DESCARTE ───
    lines += [
        "# Execution Report — 02m_user_items (DESCARTE)",
        "",
        f"**Notebook:** notebooks/fase1_cleaning/02m_user_items.ipynb",
        f"**Fecha:** {now_str}",
        f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
        f"**CSV de entrada:** data/data_raw/user_items.csv (2.08 GB, {n_total:,} filas, 11 cols)",
        f"**Parquet de salida:** NO GENERADO — descarte tras diagnóstico",
        "",
        "---", "",
        "## Resumen",
        "",
        "La tabla `user_items.csv` se descarta de Fase 1 tras diagnóstico exhaustivo. "
        "El motivo se detalla abajo. La decisión también se registra en "
        "`informes/decision_descarte.md`.",
        "",
        "---", "",
        "## Diagnóstico que motiva el descarte",
        "",
        "| Métrica | Valor | Umbral |",
        "|---|---|---|",
        f"| Cobertura del sample | {coverage_sample*100:.2f}% | <30% (umbral C) |",
        f"| `user_id` no nulo (% global) | {n_nonnull_uid/n_total*100:.2f}% | — |",
        f"| Stats `c_base_*` 100% nulas en sample | {stats_all_null} | — |",
        f"| `tempering_level` % nulo en sample | {tempering_pct_null:.2f}% | — |",
        "",
        f"**Escenario detectado: {ESCENARIO} ({desc_escenario})**",
        "",
        "---", "",
        "## Naturaleza de la tabla detectada",
        "",
        f"- Filas totales: {n_total:,}",
        f"- Filas con user_id poblado: {n_nonnull_uid:,} ({n_nonnull_uid/n_total*100:.2f}%)",
        f"- Filas con user_id NaN: {n_null_uid:,} ({n_null_uid/n_total*100:.2f}%)",
        f"- Meses únicos en CSV: {len(all_months)} (rango {all_months[0]} → {all_months[-1]})",
        "",
        "Distribución temporal (top 10 meses):",
        "",
        "| Mes | Con user_id | Sin user_id | % con user_id |",
        "|---|---|---|---|",
    ]
    for m, total in top_months:
        with_uid = month_counts_with_uid.get(m, 0)
        no_uid = month_counts_no_uid.get(m, 0)
        pct = with_uid / total * 100 if total else 0
        lines.append(f"| {m} | {with_uid:,} | {no_uid:,} | {pct:.1f}% |")

    lines += [
        "",
        "---", "",
        "## Reversibilidad",
        "",
        "Decisión revisable si:",
        "- Se obtiene una versión del CSV con `user_id` poblado en >30% del sample.",
        "- Se identifica un script de extracción que filtre por sample antes del export.",
        "- Las stats `c_base_*` se confirman pobladas en una versión futura del export.",
        "",
        "---", "",
        "## TODO para 02z (master table)",
        "",
        "Ninguno. Sin cobertura suficiente no procede ni siquiera derivar un flag binario.",
        "",
        "---", "",
        "## Acción tomada",
        "",
        f"- NO se genera `data/data_qc/features_user_items_cutoff.parquet`",
        f"- Append entry a `informes/decision_descarte.md` (sección user_items)",
        "",
        "---", "",
        "## Pasos ejecutados", "",
    ]
    for s in steps_log:
        lines.append(f"- {s}")
    lines += [
        "",
        "---", "",
        "## Constantes utilizadas",
        "",
        "| Constante | Valor |",
        "|---|---|",
        f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
        f"| `N_SAMPLE` | {N_SAMPLE:,} |",
        f"| `CHUNKSIZE` | {CHUNKSIZE:,} |",
        f"| `COV_THRESHOLD_A` | {COV_THRESHOLD_A:.2f} |",
        f"| `COV_THRESHOLD_B` | {COV_THRESHOLD_B:.2f} |",
        "",
    ]
else:
    # ─── Plantilla A/B (procesamiento completo) ───
    n_users_with_items_final = int((features['items_total_instances'] > 0).sum())
    n_users_no_items_final = N_SAMPLE - n_users_with_items_final

    lines += [
        f"# Execution Report — 02m_user_items (escenario {ESCENARIO})",
        "",
        f"**Notebook:** notebooks/fase1_cleaning/02m_user_items.ipynb",
        f"**Fecha:** {now_str}",
        f"**Tiempo de ejecucion:** {t_min} min {t_sec}s",
        f"**CSV de entrada:** data/data_raw/user_items.csv (2.08 GB, {n_total:,} filas, {N_COLS_RAW} cols)",
        f"**Parquet de salida:** data/data_qc/features_user_items_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]} cols)",
        "",
        "---", "",
        "## Resumen",
        "",
        f"- **Tabla origen**: `user_items.csv` ({n_total:,} filas, 2.08 GB)",
        f"- **Filas con user_id (no catálogo)**: {n_nonnull_uid:,} ({n_nonnull_uid/n_total*100:.2f}%)",
        f"- **Filas tras filtrar al sample**: {n_in_sample:,}",
        f"- **Sample size**: {N_SAMPLE:,} jugadores",
        f"- **Coverage del sample**: {coverage_sample*100:.2f}% ({n_users_with_items_final:,} con ≥1 ítem)",
        f"- **Escenario detectado**: {ESCENARIO} ({desc_escenario})",
        f"- **REFERENCE_DATE**: {REFERENCE_DATE}",
        "",
        "---", "",
        "## Diagnóstico previo (D1-D4)",
        "",
        "### D1 — Cobertura global de `user_id`",
        f"- Filas totales: {n_total:,}",
        f"- Con user_id: {n_nonnull_uid:,} ({n_nonnull_uid/n_total*100:.2f}%)",
        f"- Sin user_id (catálogo): {n_null_uid:,} ({n_null_uid/n_total*100:.2f}%)",
        "",
        "### D2 — Distribución temporal (top 10 meses)",
        "",
        "| Mes | Con user_id | Sin user_id | % con user_id |",
        "|---|---|---|---|",
    ]
    for m, total in top_months:
        with_uid = month_counts_with_uid.get(m, 0)
        no_uid = month_counts_no_uid.get(m, 0)
        pct = with_uid / total * 100 if total else 0
        lines.append(f"| {m} | {with_uid:,} | {no_uid:,} | {pct:.1f}% |")

    lines += [
        "",
        "### D3 — Cobertura del sample",
        f"- Usuarios del sample con ≥1 ítem: {n_users_with_items_final:,}/{N_SAMPLE:,} ({coverage_sample*100:.2f}%)",
        "",
        "### D4 — Estado de columnas",
        f"- Stats `c_base_*` 100% nulas: {stats_all_null}",
        f"- `tempering_level` % nulo (sample): {tempering_pct_null:.2f}%",
        f"- {desc_tempering}",
        "",
        "---", "",
        "## Hipótesis validadas",
        "",
        "| Hipótesis | Resultado | Estado |",
        "|---|---|---|",
        f"| H1 — Primeras N filas son catálogo (user_id NaN) | {n_null_uid:,} filas catálogo ({n_null_uid/n_total*100:.2f}%) | confirmada |",
        f"| H2 — `tempering_level` 100% nulo en toda la tabla | {tempering_pct_null:.2f}% nulo en sample | {'refutada' if tempering_pct_null < 100.0 else 'confirmada'} |",
        f"| H3 — Cobertura sample ≥70% (escenario A) | {coverage_sample*100:.2f}% | {'A' if ESCENARIO == 'A' else ('B' if ESCENARIO == 'B' else 'C')} |",
        f"| H4 — `updated_at` ≡ `created_at` | {H4_VALUE:.4f}% | {H4_STATE} |",
        "| H5 — Correlación pct_max_enhanced ↔ engagement | (a validar en EDA Fase 2) | pendiente |",
        "",
        "---", "",
        "## Constantes utilizadas",
        "",
        "| Constante | Valor |",
        "|---|---|",
        f"| `REFERENCE_DATE` | {REFERENCE_DATE} |",
        f"| `SENTINEL_DAYS` | {SENTINEL_DAYS} |",
        f"| `N_SAMPLE` | {N_SAMPLE:,} |",
        f"| `CHUNKSIZE` | {CHUNKSIZE:,} |",
        f"| `COV_THRESHOLD_A` / `COV_THRESHOLD_B` | {COV_THRESHOLD_A:.2f} / {COV_THRESHOLD_B:.2f} |",
        "",
        "---", "",
        "## Pasos ejecutados", "",
    ]
    for s in steps_log:
        lines.append(f"- {s}")

    lines += [
        "",
        "---", "",
        "## Filtrado aplicado",
        "",
        "| Paso | Filas | % CSV |",
        "|---|---|---|",
        f"| CSV original | {n_total:,} | 100.00% |",
        f"| Filas con user_id (no catálogo) | {n_nonnull_uid:,} | {n_nonnull_uid/n_total*100:.2f}% |",
        f"| En sample | {n_in_sample:,} | {n_in_sample/n_total*100:.2f}% |",
        "",
        "---", "",
        "## Features generadas (13 + user_id)",
        "",
        "| Feature | Tipo | Definición |",
        "|---|---|---|",
        "| `items_total_instances` | int32 | `count(*)` por user_id (instancias) |",
        "| `items_unique_definitions` | int32 | `n_unique(item_definition_excel_id)` |",
        "| `items_redundancy_ratio` | float32 | `total/unique` (NaN si sin ítems) |",
        "| `items_mean_enhance_level` | float32 | `mean(enhance_level)` ignorando NaN |",
        "| `items_max_enhance_level` | float32 | `max(enhance_level)` |",
        "| `items_total_enhance_invested` | int32 | `sum(enhance_level)` |",
        "| `items_mean_attack` | float32 | `mean(c_base_attack_enhanced)` |",
        "| `items_mean_defense` | float32 | `mean(c_base_defense_enhanced)` |",
        "| `items_mean_critical` | float32 | `mean(c_base_critical_chance)` |",
        "| `items_attack_defense_ratio` | float32 | `mean_attack / mean_defense` (NaN si denom 0) |",
        "| `items_first_item_at` | datetime64[ns, UTC] | `min(created_at)` (NaT si sin ítems) |",
        "| `items_last_item_at` | datetime64[ns, UTC] | `max(created_at)` (NaT si sin ítems) |",
        f"| `items_days_since_last_item` | int32 | `(CUTOFF_DATE - last).days` (`{SENTINEL_DAYS}` sentinel) |",
        "",
        "**Features descartadas / aplazadas:**",
        "",
        "- `items_mean_tempering`: NO generada (cuasi-constante en valor 1.0 — std=0.05 en run inicial).",
        "- `items_n_max_enhanced` y `items_pct_max_enhanced`: NO generadas (97.9% del sample con valor 0 — sin varianza útil).",
        "- Top-K `item_definition_excel_id`: aplazado a Fase 2 (cardinalidad alta sin catálogo).",
        f"- Features sobre `updated_at`: NO generadas (H4 {'confirmada — equivalente' if H4_STATE == 'validada' else 'refutada — pendiente Fase 2'}).",
        "",
        "---", "",
        "## Sanity checks aplicados (8)",
        "",
        f"- [x] Shape == ({N_SAMPLE:,}, 14)",
        "- [x] user_id único",
        "- [x] `items_unique_definitions <= items_total_instances`",
        "- [x] `items_redundancy_ratio >= 1` cuando definido",
        "- [x] `items_max_enhance_level >= items_mean_enhance_level`",
        f"- [x] `items_days_since_last_item >= 0` (excluye SENTINEL={SENTINEL_DAYS})",
        f"- [x] `items_days_since_last_item == {SENTINEL_DAYS}` ↔ `items_last_item_at` NaT",
        "- [x] Tipos correctos (int32×4, float32×7, datetime×2 + str user_id)",
        "",
        "---", "",
        "## Decisiones tomadas",
        "",
        "1. **2 pasadas chunked** (`CHUNKSIZE=500_000`): pasada 1 lightweight con solo `user_id`+`created_at` para diagnóstico cobertura. Pasada 2 con `USECOLS` completo solo si A/B. Evita OOM y procesamiento innecesario en escenario C.",
        f"2. **Lógica A/B/C explícita**: umbrales {COV_THRESHOLD_A:.0%} / {COV_THRESHOLD_B:.0%} sobre cobertura sample. Stats `c_base_*` 100% nulas también dispara C.",
        f"3. **`tempering_level`**: {desc_tempering}.",
        f"4. **Features sobre `updated_at`**: NO generadas (H4 {H4_STATE}).",
        "5. **Top-K categorías aplazadas**: por alta cardinalidad sin catálogo de items.",
        f"6. **SENTINEL_DAYS = {SENTINEL_DAYS}** para `items_days_since_last_item` (consistente con 02g, 02l).",
        "7. **`user_id` forzado a `dtype=str`** en `read_csv`: el dtype-inference lo detecta como float64 por las primeras filas todas-NaN (catálogo); forzar str preserva el hex original.",
        "",
        "---", "",
        "## Lo que ha ido bien",
        "",
        f"- 2 pasadas chunked en {load_seconds + t_pass1:.0f}s totales (pico {peak_chunk_mem:.0f} MB).",
        f"- Detección automática de la sección catálogo ({n_null_uid/n_total*100:.1f}% del CSV) sin asunciones rígidas sobre fila de corte.",
        f"- Cobertura del sample del {coverage_sample*100:.2f}% (escenario {ESCENARIO}).",
        "",
        "---", "",
        "## Limitaciones y caveats",
        "",
        f"- **Sección catálogo descartada**: {n_null_uid:,} filas ({n_null_uid/n_total*100:.2f}%) sin `user_id` se ignoran. Probablemente es la semilla del juego (fechas tempranas).",
        "- **Sin catálogo de items**: las stats agregadas por usuario no se pueden contextualizar por tipo de ítem hasta Fase 2.",
        f"- **Jugadores sin ítems**: {n_users_no_items_final:,} ({n_users_no_items_final/N_SAMPLE*100:.2f}%) → counts a 0, ratios y means a NaN, `items_days_since_last_item={SENTINEL_DAYS}`.",
        "- **`items_attack_defense_ratio`** sensible a denominador 0 → NaN explícita.",
        "",
        "---", "",
        "## Archivos generados",
        "",
        "| Archivo | Descripción |",
        "|---|---|",
        f"| features_user_items_cutoff.parquet ({features.shape[1]} cols) | Parquet final |",
        "| nulos_por_columna_raw.csv | Nulos en subset filtrado pre-agregación |",
        "| nulos_features_final.csv | Nulos en features finales |",
        "| features_describe.csv | Estadísticas descriptivas |",
        "| features_distributions.png | Histogramas (log scale donde aplica) |",
        "| sample_head.csv | 20 primeras filas del parquet |",
        "| 02m_user_items_run.html | Notebook ejecutado completo (logging dual) |",
        "| execution_report.md | Este informe |",
        "",
        "---", "",
        "## Advertencias para notebooks siguientes",
        "",
        f"- `items_days_since_last_item == {SENTINEL_DAYS}` identifica usuarios sin instancias.",
        "- `items_attack_defense_ratio` puede ser NaN: imputar en 02z según modelo (1.0 = neutral, NaN-flag para árboles).",
        "- `items_*` y `coll_*` (de 02l) son **complementarios**, no redundantes: amplitud (collection) vs profundidad (items).",
        "",
        "---", "",
        "## TODOs para fases siguientes",
        "",
        "- **Fase 2 (EDA)**: validar H5 (correlación pct_max_enhanced ↔ engagement/churn).",
        "- **Fase 2**: cruzar `items_attack_defense_ratio` con `arena_log` outcomes (¿builds ofensivas ganan más?).",
        "- **Fase 3 (modelado)**: log-transform sobre `items_total_instances`, `items_total_enhance_invested` (sesgo extremo).",
        "- **Modelo de gustos**: `items_attack_defense_ratio` y `items_mean_critical` son candidatos de alto valor para segmentación de estilos.",
        "- **Catálogo de items**: si llega, derivar `items_top_family_*`.",
        "",
        "---", "",
        "## Diagrama del pipeline",
        "",
        "```",
        f"user_items.csv ({n_total:,} filas × {N_COLS_RAW} cols, 2.08 GB)",
        "    │",
        "    ├─ Pasada 1 (lightweight): user_id + created_at",
        f"    │   └─ D1 cobertura, D2 distribución temporal",
        "    │",
        "    ├─ Decisión A/B/C → escenario {ESCENARIO}",
        "    │",
        f"    └─ Pasada 2 (USECOLS): drop _id, dtype user_id=str   (-{n_total - n_in_sample:,})",
        "        ├─ Filtrar user_id no nulo + en sample",
        "        ├─ D3 cobertura sample, D4 nulos por columna",
        "        └─ Parsear created_at, updated_at",
        "",
        f"Filtrado ({n_in_sample:,} filas)",
        "    │",
        "    ├─ Validar H4",
        "    ├─ Agregaciones por user_id (groupby + comparación intra-fila)",
        "    │   └─ count, nunique, sum, min/max, mean (×3 stats)",
        "    └─ Reindex sample + fillna/sentinel",
        "",
        f"features_user_items_cutoff.parquet ({features.shape[0]:,} × {features.shape[1]})",
        "```",
        "",
        "---", "",
        "## Reproducibilidad",
        "",
        "1. Ejecutar 02a_users.ipynb primero (genera sample_user_ids)",
        "2. Ejecutar 02m_user_items.ipynb",
        f"3. Verificar: features_user_items_cutoff.parquet == 126.253 filas × {features.shape[1]} cols",
        "",
        "---", "",
        "## Referencias",
        "",
        "- PLANTILLA_NOTEBOOK_02.md — estructura estándar de notebook Fase 1.",
        "- 02l_user_items_collection.ipynb — tabla complementaria (amplitud vs profundidad).",
        "- 02g_devices.ipynb — referencia para SENTINEL_DAYS.",
        "",
    ]

# Guardar el report
report_path = REPORT_DIR / 'execution_report.md'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f"Informe guardado: {report_path}")
log_step('REPORT', f'execution_report.md generado (escenario {ESCENARIO})')


Informe guardado: /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_items/execution_report.md
[REPORT] 13:14:33 — execution_report.md generado (escenario A)


In [16]:
# [REPORT] Si escenario C, append a informes/decision_descarte.md
if ESCENARIO != 'C':
    print(f'[Saltado — escenario {ESCENARIO}: tabla procesada, no se descarta]')
else:
    if not DESCARTE_PATH.exists():
        print(f"{DESCARTE_PATH} no existe — el report de descarte de 02m queda solo en su carpeta.")
    else:
        entry_lines = [
            "",
            "## user_items.csv",
            "",
            f"- **Fecha de descarte**: {datetime.now().strftime('%Y-%m-%d')}",
            f"- **Notebook planificado**: `02m_user_items` (NO genera parquet — descartado tras diagnóstico)",
            f"- **Motivo**: cobertura del sample <{COV_THRESHOLD_B*100:.0f}% (real: {coverage_sample*100:.2f}%) "
            f"y/o stats `c_base_*` 100% nulas en subset filtrado.",
            "",
            "### Justificación cuantitativa",
            "",
            "| Métrica | Valor |",
            "|---|---|",
            f"| Filas totales del CSV | {n_total:,} |",
            f"| Filas con `user_id` poblado | {n_nonnull_uid:,} ({n_nonnull_uid/n_total*100:.2f}%) |",
            f"| Cobertura del sample | {coverage_sample*100:.2f}% |",
            f"| Stats `c_base_*` 100% nulas | {stats_all_null} |",
            f"| `tempering_level` % nulo (sample) | {tempering_pct_null:.2f}% |",
            "",
            "### Reversibilidad",
            "",
            "Decisión revisable si se obtiene una versión del CSV con `user_id` "
            "poblado en >30% de las filas correspondientes al sample. Mientras tanto, "
            "no entra en el pipeline de Fase 1.",
            "",
            "### TODO para 02z",
            "",
            "Ninguno — sin cobertura suficiente no procede ni siquiera derivar un flag binario.",
            "",
            "---",
        ]
        with open(DESCARTE_PATH, 'a', encoding='utf-8') as f:
            f.write('\n'.join(entry_lines))
        print(f"Append a {DESCARTE_PATH} hecho")
        log_step('REPORT', 'decision_descarte.md actualizado')


[Saltado — escenario A: tabla procesada, no se descarta]


In [17]:
# [REPORT] Resumen final en consola
elapsed = time.time() - NOTEBOOK_START

print("=" * 70)
print(f"RESUMEN FINAL — Notebook 02m_user_items (escenario {ESCENARIO})")
print("=" * 70)
print(f"  Tiempo total                : {int(elapsed//60)}m {int(elapsed%60)}s")
print(f"  Filas CSV original          : {n_total:,}")
print(f"  Filas con user_id           : {n_nonnull_uid:,} ({n_nonnull_uid/n_total*100:.2f}%)")
print(f"  Filas en sample             : {n_in_sample:,}")
print(f"  Cobertura del sample        : {coverage_sample*100:.2f}% (umbral A={COV_THRESHOLD_A*100:.0f}%, B={COV_THRESHOLD_B*100:.0f}%)")
print(f"  Stats c_base_* todas nulas  : {stats_all_null}")
print(f"  tempering_level %nulo       : {tempering_pct_null:.2f}%")
print(f"  ESCENARIO                   : {ESCENARIO}  ({desc_escenario})")
print()
if ESCENARIO == 'C':
    print(f"  PARQUET NO GENERADO — tabla descartada")
    print(f"  Report de descarte: {REPORT_DIR / 'execution_report.md'}")
    if DESCARTE_PATH.exists():
        print(f"  Append a {DESCARTE_PATH}")
else:
    n_users_with_items_final = int((features['items_total_instances'] > 0).sum())
    print(f"  Filas features final        : {len(features):,} (== {N_SAMPLE:,} sample)")
    print(f"  Columnas features           : {features.shape[1]}")
    print(f"  Usuarios con ítems          : {n_users_with_items_final:,}")
    print()
    print(f"  items_total_instances       : mean={features['items_total_instances'].mean():.1f}, max={features['items_total_instances'].max():,}, p99={features['items_total_instances'].quantile(0.99):.0f}")
    print(f"  items_unique_definitions    : mean={features['items_unique_definitions'].mean():.1f}, max={features['items_unique_definitions'].max():,}")
    print(f"  items_mean_enhance_level    : mean={features['items_mean_enhance_level'].mean():.2f}")
    print(f"  items_total_enhance_invested: mean={features['items_total_enhance_invested'].mean():.1f}")
    print(f"  items_attack_defense_ratio  : mean={features['items_attack_defense_ratio'].mean():.3f}, NaN={int(features['items_attack_defense_ratio'].isna().sum()):,}")
    print()
    print(f"  H4: updated_at==created_at = {H4_VALUE:.4f}% ({H4_STATE})")
    print()
    print("Archivos generados:")
    print(f"  features_user_items_cutoff.parquet ({PARQUET_FEATURES.stat().st_size/1e6:.2f} MB)")
    print(f"  execution_report.md")
    print(f"  Gráficos y CSVs en {REPORT_DIR}")
print("=" * 70)


RESUMEN FINAL — Notebook 02m_user_items (escenario A)
  Tiempo total                : 0m 30s
  Filas CSV original          : 17,450,439
  Filas con user_id           : 10,636,591 (60.95%)
  Filas en sample             : 720,477
  Cobertura del sample        : 100.00% (umbral A=70%, B=30%)
  Stats c_base_* todas nulas  : False
  tempering_level %nulo       : 0.00%
  ESCENARIO                   : A  (Procesable — cobertura ≥70% y stats pobladas)

  Filas features final        : 25,200 (== 25,200 sample)
  Columnas features           : 14
  Usuarios con ítems          : 24,366

  items_total_instances       : mean=22.1, max=471, p99=133
  items_unique_definitions    : mean=17.0, max=144
  items_mean_enhance_level    : mean=2.50
  items_total_enhance_invested: mean=41.5
  items_attack_defense_ratio  : mean=1.127, NaN=834

  H4: updated_at==created_at = 65.8465% (refutada)

Archivos generados:
  features_user_items_cutoff.parquet (1.73 MB)
  execution_report.md
  Gráficos y CSVs en /Users/j

In [18]:
# [REPORT] Logging dual: HTML + sección enriquecida del report
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase1_cleaning' / '02m_user_items.ipynb'
html_path = REPORT_DIR / '02m_user_items_run.html'
export_notebook_to_html(notebook_path, html_path)

if ESCENARIO != 'C':
    enriched = build_enriched_report_section(
        df_final=features,
        raw_shape=(n_total, N_COLS_RAW),
        filter_steps=[
            ('CSV original', n_total),
            ('Con user_id (no catálogo)', n_nonnull_uid),
            ('En sample', n_in_sample),
        ],
    )
    with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
        f.write('\n\n---\n\n' + enriched)
    print(f"Enriquecimiento appendeado a {REPORT_DIR / 'execution_report.md'}")
else:
    print(f"[Enriquecimiento omitido — escenario {ESCENARIO} no genera DataFrame final]")


HTML generado: 02m_user_items_run.html (0.5 MB)
Enriquecimiento appendeado a /Users/jezquerro/Documents/tfg/informes/fase1_cleaning/user_items/execution_report.md
